# Podział danych na poziomie pacjentów

Celem notebooka jest ustalenie i zapisanie jednego przydziału pacjentów do treningu, walidacji i testu.
Wszystkie przekroje jednego pacjenta trafiają do tej samej części.

Przyjęty podział: **77 pacjentów do treningu, 16 do walidacji, 17 do testu**, przy ziarnie losowania **42**.
Wejściem są tabele `data/metadata/patients.csv` i `data/metadata/pairs.csv` z poprzedniego notebooka;
nie odczytujemy tutaj obrazów MRI.

Uruchamiaj komórki od góry, z kernelem **Python (inzynierka)**, z katalogu projektu lub `notebooks/`.
W kolejnych eksperymentach korzystamy z zapisanego `data/splits/splits.csv`, zamiast ponownie ustalać przydziały.

## 1. Biblioteki i ścieżki

`Path` obsługuje ścieżki, NumPy służy do losowania kolejności, a Pandas do odczytu i łączenia tabel.
Katalog projektu wyznaczamy na podstawie katalogu roboczego kernela, bez lokalnej ścieżki użytkownika.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

metadata_dir = project_root / "data" / "metadata"
split_dir = project_root / "data" / "splits"

## 2. Lista pacjentów i powtarzalne losowanie

Tabela `patient_df` ma jeden wiersz na pacjenta. Najpierw oglądamy jej początek.

In [3]:
patient_df = pd.read_csv(metadata_dir / "patients.csv")
patient_df.head()

,patient_id,n_slices,n_annotated_slices
0,TCGA_CS_4941_19960909,23,8
1,TCGA_CS_4942_19970222,20,6
2,TCGA_CS_4943_20000902,20,8
3,TCGA_CS_4944_20010208,20,9
4,TCGA_CS_5393_19990606,20,8


Przed losowaniem sortujemy identyfikatory, aby kolejność wejściowa była stała.
`default_rng(42)` tworzy generator, a `permutation(...)` zwraca przetasowaną kopię identyfikatorów.
Generator i losowanie są w jednej komórce, więc ponowne jej wykonanie zaczyna od tego samego stanu.
Kontrolujemy liczbę identyfikatorów i ich unikalność; oczekujemy 110 i 110.

In [4]:
patient_ids = sorted(patient_df["patient_id"])
rng = np.random.default_rng(42)
shuffled_ids = rng.permutation(patient_ids)

print("Liczba identyfikatorów:", len(patient_ids))
print("Unikalni pacjenci:", patient_df["patient_id"].nunique())
shuffled_ids[:5]


Liczba identyfikatorów: 110
Unikalni pacjenci: 110


array(['TCGA_HT_7694_19950404', 'TCGA_DU_5852_19950709',
       'TCGA_DU_7019_19940908', 'TCGA_FG_8189_20030516',
       'TCGA_HT_7686_19950629'], dtype='<U21')

## 3. Przydział 77 / 16 / 17

Z przetasowanej tablicy wybieramy pierwszych 77 pacjentów do treningu, kolejnych 16 do walidacji,
a pozostałych 17 do testu. Prawy koniec przedziału `[start:stop]` nie jest uwzględniany.

In [5]:
train_ids = shuffled_ids[:77]
val_ids = shuffled_ids[77:77 + 16]
test_ids = shuffled_ids[77 + 16:]

print("Trening:", len(train_ids))
print("Walidacja:", len(val_ids))
print("Test:", len(test_ids))
print("Razem:", len(train_ids) + len(val_ids) + len(test_ids))


Trening: 77
Walidacja: 16
Test: 17
Razem: 110


## 4. Rozłączność i kompletność

`isdisjoint(...)` sprawdza brak wspólnych pacjentów między każdą parą części.
Suma zbiorów przez `|` musi być równa pełnej liście pacjentów: nikogo nie pomijamy i nikogo nie dodajemy.
Oczekujemy czterech wyników `True`.

In [6]:
print("Trening i walidacja rozłączne:", set(train_ids).isdisjoint(set(val_ids)))
print("Trening i test rozłączne:", set(train_ids).isdisjoint(set(test_ids)))
print("Walidacja i test rozłączne:", set(val_ids).isdisjoint(set(test_ids)))
print(
    "Przydzielono dokładnie wszystkich pacjentów:",
    (set(train_ids) | set(val_ids) | set(test_ids)) == set(patient_ids),
)


Trening i walidacja rozłączne: True
Trening i test rozłączne: True
Walidacja i test rozłączne: True
Przydzielono dokładnie wszystkich pacjentów: True


## 5. Tabela przydziałów i zapis CSV

Każdy wiersz `split_df` zawiera identyfikator pacjenta oraz etykietę `train`, `val` lub `test`.
`pd.concat(..., ignore_index=True)` łączy trzy tabele i nadaje wspólny indeks.
Oczekujemy 110 wierszy, dwóch kolumn i 110 unikalnych identyfikatorów.

In [7]:
train_split = pd.DataFrame({
    "patient_id": train_ids,
    "split": "train",
})

val_split = pd.DataFrame({
    "patient_id": val_ids,
    "split": "val",
})

test_split = pd.DataFrame({
    "patient_id": test_ids,
    "split": "test",
})

split_df = pd.concat([train_split, val_split, test_split], ignore_index=True)
print("Kształt tabeli:", split_df.shape)
print("Liczba pacjentów w częściach:")
print(split_df["split"].value_counts())
print("Unikalni pacjenci:", split_df["patient_id"].nunique())
split_df.head()


Kształt tabeli: (110, 2)
Liczba pacjentów w częściach:
split
train    77
test     17
val      16
Name: count, dtype: int64
Unikalni pacjenci: 110


,patient_id,split
0,TCGA_HT_7694_19950404,train
1,TCGA_DU_5852_19950709,train
2,TCGA_DU_7019_19940908,train
3,TCGA_FG_8189_20030516,train
4,TCGA_HT_7686_19950629,train


Zapisujemy konkretne przydziały w `data/splits/splits.csv`; sam seed nie zastępuje tego pliku.
`index=False` pomija techniczny indeks DataFrame. Ponowne wykonanie tej komórki nadpisuje CSV
aktualną tabelą; katalog `data/splits/` powinien już istnieć.

In [8]:
split_df.to_csv(split_dir / "splits.csv", index=False)

Odczytujemy zapisany plik do `splits_loaded` i sprawdzamy jego rozmiar, kolumny oraz zgodność z tabelą w pamięci.
Oczekiwany wynik to `(110, 2)`, kolumny `patient_id` i `split` oraz `True` z porównania `.equals(...)`.

In [9]:
splits_loaded = pd.read_csv(split_dir / "splits.csv")
print("Kształt po odczycie:", splits_loaded.shape)
print("Kolumny:", splits_loaded.columns.tolist())
print("Zgodność z tabelą przed zapisem:", splits_loaded.equals(split_df))


Kształt po odczycie: (110, 2)
Kolumny: ['patient_id', 'split']
Zgodność z tabelą przed zapisem: True


## 6. Przypisanie części do przekrojów

Łączymy tabelę par z **odczytanym plikiem przydziałów** według `patient_id`.
`how="left"` zachowuje wszystkie pary, a `validate="many_to_one"` sprawdza,
czy wiele przekrojów łączymy z najwyżej jednym przydziałem danego pacjenta.
Kontrola braków w `split` potwierdza, czy każdy przekrój otrzymał przydział.
Oczekujemy `(3929, 8)` i zera braków.

In [10]:
pairs_df = pd.read_csv(metadata_dir / "pairs.csv")
pairs_with_split = pairs_df.merge(
    splits_loaded,
    on="patient_id",
    how="left",
    validate="many_to_one",
)
print("Kształt tabeli par z przydziałami:", pairs_with_split.shape)
print("Przekroje bez przydziału:", pairs_with_split["split"].isna().sum())
pairs_with_split.head()


Kształt tabeli par z przydziałami: (3929, 8)
Przekroje bez przydziału: 0


,patient_id,image_path,mask_path,correct_shape,slice_id,foreground_pixels,has_annotation,split
0,TCGA_CS_4941_19960909,data/raw/TCGA_CS_4941_19960909/TCGA_CS_4941_19...,data/raw/TCGA_CS_4941_19960909/TCGA_CS_4941_19...,True,1,0,False,train
1,TCGA_CS_4941_19960909,data/raw/TCGA_CS_4941_19960909/TCGA_CS_4941_19...,data/raw/TCGA_CS_4941_19960909/TCGA_CS_4941_19...,True,2,0,False,train
2,TCGA_CS_4941_19960909,data/raw/TCGA_CS_4941_19960909/TCGA_CS_4941_19...,data/raw/TCGA_CS_4941_19960909/TCGA_CS_4941_19...,True,3,0,False,train
3,TCGA_CS_4941_19960909,data/raw/TCGA_CS_4941_19960909/TCGA_CS_4941_19...,data/raw/TCGA_CS_4941_19960909/TCGA_CS_4941_19...,True,4,0,False,train
4,TCGA_CS_4941_19960909,data/raw/TCGA_CS_4941_19960909/TCGA_CS_4941_19...,data/raw/TCGA_CS_4941_19960909/TCGA_CS_4941_19...,True,5,0,False,train


Podsumowujemy liczbę pacjentów i przekrojów w każdej części.
Pacjenci mają różną liczbę obrazów, więc proporcje przekrojów mogą różnić się od proporcji pacjentów.
Dla ustalonego przydziału otrzymaliśmy: trening 77 / 2714, walidacja 16 / 615, test 17 / 600
(pacjenci / przekroje). Podsumowanie opisuje wynik losowania; nie służy do ponownego dobierania przydziału.

In [11]:
split_summary = pairs_with_split.groupby("split").agg(
    n_patients=("patient_id", "nunique"),
    n_slices=("slice_id", "count"),
)
split_summary


,n_patients,n_slices
split,,
test,17,600
train,77,2714
val,16,615


## 7. Część treningowa — punkt startowy dalszej analizy

Wybieramy trening i tworzymy osobną kopię tabeli przez `.copy()`.
Kontrolujemy liczbę przekrojów, pacjentów i przekrojów z oznaczeniem.
Wynik: 2714 przekrojów od 77 pacjentów, w tym 922 z oznaczeniem i 1792 z pustą maską.
Są to liczby przekrojów, a nie proporcje klas na poziomie pikseli.

In [12]:
train_pairs = pairs_with_split[pairs_with_split["split"] == "train"].copy()
print("Kształt części treningowej:", train_pairs.shape)
print("Pacjenci treningowi:", train_pairs["patient_id"].nunique())
print("Przekroje według obecności oznaczenia:")
print(train_pairs["has_annotation"].value_counts())


Kształt części treningowej: (2714, 8)
Pacjenci treningowi: 77
Przekroje według obecności oznaczenia:
has_annotation
False    1792
True      922
Name: count, dtype: int64
